# Experiment E06: Complete Generalization Benchmark Suite
## TE-Q-Transformer V2 Research Project

**Scientific Research Question:**  
*"Does the observed performance of TE-Q-Transformer remain consistent when evaluated on cells that were not used during training (unseen-cell generalization) and on a temporally held-out portion of a battery cell (temporal extrapolation)?"*

---

### E06 Generalization Framework:
1. **Unseen-Cell Generalization:** Evaluated on held-out cells **B0018** (132 cycles) and **B0032** (39 cycles). These cells are strictly unseen during training, feature scaling, model selection, and hyperparameter tuning.
2. **Temporal Extrapolation:** Evaluated on the final 30% segment of **B0053** (cycles 37–52, 16 cycles). The first 70% of B0053 (cycles 0–36, 37 cycles) is part of the training pool.
3. **Scientific Terminology Requirement:** E06 is **NOT** an "unseen-temperature" experiment. Temperature in NASA Ames is coupled with operating conditions; evaluations are strictly designated as `unseen_cell` and `temporal_extrapolation`.

---

### Complete Benchmark Suite (All 10 Baselines from baselineComparison.ipynb + Proposed TE-Q-Transformer):
1. **TE-Q-Transformer** (Proposed physics-guided quantum model; 92,554 parameters)
2. **LSTM** (Classical Recurrent baseline; 71,105 parameters)
3. **GRU** (Lightweight Classical Recurrent baseline; 54,465 parameters)
4. **CNN1D** (1D Dilated Convolutional baseline; 47,041 parameters)
5. **TCN** (Temporal Convolutional Network; 91,841 parameters)
6. **DLinear** (Direct Linear Decomposition baseline; 1,031 parameters)
7. **Transformer** (Classical multi-head attention encoder; 80,257 parameters)
8. **PatchTST** (Channel-independent patch Transformer; 109,962 parameters)
9. **iTransformer** (Inverted variate token Transformer; 137,473 parameters)
10. **QLSTM** (Hybrid QNN front-end + LSTM; 37,853 parameters)
11. **QNN-GRU** (Quantum Neural Network feature layer + GRU baseline; 29,533 parameters)


In [ ]:
# ==============================================================================
# SECTION 0: ENVIRONMENT, HARDWARE DETECTION, REPRODUCIBILITY & METRICS
# ==============================================================================
import os
import sys
import gc
import time
import math
import json
import random
import platform
import hashlib
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Tuple, Callable

# Auto-install PennyLane if missing (Kaggle/Colab runtimes)
try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    os.system(f"{sys.executable} -m pip install -q pennylane")
    import pennylane as qml

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deterministic Seed Configuration
SEEDS = [42]
CURRENT_SEED = SEEDS[0]

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CURRENT_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Execution Mode:
# Set RUN_MODE = 'DRY_RUN' for fast 1-epoch verification.
# Set RUN_MODE = 'FULL' for complete 80-epoch publication benchmark on GPU.
RUN_MODE = "DRY_RUN"

# Output Directory Structure
OUTPUT_ROOT = Path("E06_results")
CHECKPOINTS_DIR = OUTPUT_ROOT / "checkpoints"
METRICS_DIR = OUTPUT_ROOT / "metrics"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
TRAINING_DIR = OUTPUT_ROOT / "training"
CONFIGS_DIR = OUTPUT_ROOT / "configs"
REPORTS_DIR = OUTPUT_ROOT / "reports"
PROVENANCE_DIR = OUTPUT_ROOT / "provenance"

for d in [CHECKPOINTS_DIR, METRICS_DIR, PREDICTIONS_DIR, TRAINING_DIR, CONFIGS_DIR, REPORTS_DIR, PROVENANCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Standardized Metric Evaluation Function
def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    actual = np.asarray(actual, dtype=np.float64).reshape(-1)
    predicted = np.asarray(predicted, dtype=np.float64).reshape(-1)
    abs_error = np.abs(actual - predicted)
    denom = np.clip(np.abs(actual), a_min=1e-8, a_max=None)
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae = float(mean_absolute_error(actual, predicted))
    mape = float(np.mean(abs_error / denom) * 100.0)
    max_e = float(np.max(abs_error))
    if np.allclose(actual, actual[0]):
        r2 = float("nan")
    else:
        r2 = float(r2_score(actual, predicted))
    return {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2, "MaxE": max_e}

print("=" * 75)
print(f"EXPERIMENT E06: GENERALIZATION BENCHMARK (11 MODELS)")
print(f"RUN_MODE: {RUN_MODE} | DEVICE: {DEVICE} | SEED: {CURRENT_SEED}")
print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__} | PennyLane: {qml.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Artifacts output path: {OUTPUT_ROOT.resolve()}")
print("=" * 75)


In [ ]:
# ==============================================================================
# SECTION 1: DATASET DISCOVERY & VALIDATION
# ==============================================================================
CANDIDATE_DATA_PATHS = [
    Path("data/processed/nasa_randomized"),
    Path("../input/nasa-battery-dataset/data/processed/nasa_randomized"),
    Path("../input/nasa-battery-data/data/processed/nasa_randomized"),
    Path("../input/nasa-battery-dataset"),
    Path("../input/nasa_randomized"),
    Path("/kaggle/input/nasa-battery-dataset/data/processed/nasa_randomized"),
    Path("/kaggle/input/nasa-battery-dataset"),
    Path("/kaggle/input/nasa-battery-data"),
    Path("E:/TE-Q-Transformer/TE-Q-Transformer/data/processed/nasa_randomized"),
]

DATA_ROOT = None
for p in CANDIDATE_DATA_PATHS:
    if p.exists() and (p / "B0005_X.npy").exists():
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    for search_root in [Path("."), Path(".."), Path("/kaggle/input")]:
        if search_root.exists():
            for found in search_root.rglob("B0005_X.npy"):
                DATA_ROOT = found.parent
                break
        if DATA_ROOT is not None:
            break

if DATA_ROOT is None:
    raise FileNotFoundError("Could not find NASA battery .npy data. Please place NASA .npy files in data/processed/nasa_randomized/ or attach Kaggle dataset.")

print(f"[Data Discovery] Resolved DATA_ROOT: {DATA_ROOT.resolve()}")

REQUIRED_CELLS = ["B0005", "B0006", "B0007", "B0018", "B0029", "B0030", "B0031", "B0032", "B0053"]
print("\n" + "=" * 75)
print(f"{'Cell':10s} | {'X Shape':18s} | {'SOH Shape':12s} | {'Initial C0 (Ah)':16s}")
print("=" * 75)

for cell_id in REQUIRED_CELLS:
    x_file = DATA_ROOT / f"{cell_id}_X.npy"
    y_file = DATA_ROOT / f"{cell_id}_soh.npy"
    if not x_file.exists():
        raise FileNotFoundError(f"Missing required data file: {x_file}")
    if not y_file.exists():
        raise FileNotFoundError(f"Missing required target file: {y_file}")
    x_arr = np.load(x_file)
    y_arr = np.load(y_file)
    print(f"{cell_id:10s} | {str(x_arr.shape):18s} | {str(y_arr.shape):12s} | {float(y_arr[0]):.4f}")

print("=" * 75)
print(f"[Validation] All {len(REQUIRED_CELLS)} NASA cell files verified successfully in DATA_ROOT.")


In [ ]:
# ==============================================================================
# SECTION 2: EXACT E05 PREPROCESSING
# ==============================================================================
NASA_FULL_TRAIN_CELLS = ("B0005", "B0006", "B0007", "B0029", "B0030", "B0031")
NASA_FULL_TEST_CELLS = ("B0018", "B0032")
NASA_SPLIT_CELL_ID = "B0053"
NASA_SPLIT_RATIO = 0.70  # First 70% train (37 cycles: 0-36), last 30% test (16 cycles: 37-52)
FEATURE_IDX_TO_SCALE = (0, 1, 3)  # Voltage, Current, Time_norm; Temp(2) left in Celsius
SEQUENCE_LENGTH = 512

class NASABatteryDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor) -> None:
        self.X = X.float()
        self.y = y.float()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

def load_cell_arrays(data_dir: Path, cell_id: str) -> Tuple[np.ndarray, np.ndarray]:
    x_path = data_dir / f"{cell_id}_X.npy"
    y_path = data_dir / f"{cell_id}_soh.npy"
    X = np.load(x_path)
    y = np.load(y_path)
    return X.astype(np.float32, copy=True), y.astype(np.float32, copy=False)

def normalize_soh_per_cell(y: np.ndarray, cell_id: str) -> np.ndarray:
    c0 = float(y[0])
    return (y / np.float32(c0)).astype(np.float32, copy=False)

def load_full_cell(data_dir: Path, cell_id: str) -> Tuple[torch.Tensor, torch.Tensor]:
    X, y = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y, cell_id)
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()

def split_cell_70_30(data_dir: Path, cell_id: str) -> Tuple[Tuple[torch.Tensor, torch.Tensor], Tuple[torch.Tensor, torch.Tensor]]:
    X, y = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y, cell_id)
    total_len = len(X)
    split_idx = int(round(total_len * NASA_SPLIT_RATIO))
    train_X = torch.from_numpy(X[:split_idx]).float()
    train_y = torch.from_numpy(y[:split_idx]).float()
    test_X = torch.from_numpy(X[split_idx:]).float()
    test_y = torch.from_numpy(y[split_idx:]).float()
    return (train_X, train_y), (test_X, test_y)

def build_feature_scaler(train_X_tensors: List[torch.Tensor]) -> MinMaxScaler:
    train_concat = torch.cat(train_X_tensors, dim=0).numpy()
    B_total, L, D = train_concat.shape
    train_reshaped = train_concat.reshape(-1, D)
    scaler = MinMaxScaler()
    scaler.fit(train_reshaped[:, list(FEATURE_IDX_TO_SCALE)])
    return scaler

def apply_feature_scaler(X: torch.Tensor, scaler: MinMaxScaler) -> torch.Tensor:
    X_np = X.numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    B, L, D = X_np.shape
    X_reshaped = X_np.reshape(-1, D).copy()
    X_reshaped[:, list(FEATURE_IDX_TO_SCALE)] = scaler.transform(X_reshaped[:, list(FEATURE_IDX_TO_SCALE)])
    return torch.from_numpy(X_reshaped.reshape(B, L, D)).float()

def get_nasa_dataloaders(
    data_dir: Path = DATA_ROOT,
    batch_size: int = 8,
    num_workers: int = 0
) -> Tuple[DataLoader, Dict[str, DataLoader], MinMaxScaler, Tuple[torch.Tensor, torch.Tensor]]:
    train_X_list, train_y_list = [], []

    for cell_id in NASA_FULL_TRAIN_CELLS:
        X, y = load_full_cell(data_dir, cell_id)
        train_X_list.append(X)
        train_y_list.append(y)

    (split_tr_X, split_tr_y), (split_te_X, split_te_y) = split_cell_70_30(data_dir, NASA_SPLIT_CELL_ID)
    train_X_list.append(split_tr_X)
    train_y_list.append(split_tr_y)

    scaler = build_feature_scaler(train_X_list)
    scaled_train_X = torch.cat([apply_feature_scaler(x, scaler) for x in train_X_list], dim=0)
    scaled_train_y = torch.cat(train_y_list, dim=0)

    train_dataset = NASABatteryDataset(scaled_train_X, scaled_train_y)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    test_loaders: Dict[str, DataLoader] = {}
    for cell_id in NASA_FULL_TEST_CELLS:
        raw_X, raw_y = load_full_cell(data_dir, cell_id)
        sc_X = apply_feature_scaler(raw_X, scaler)
        test_loaders[cell_id] = DataLoader(
            NASABatteryDataset(sc_X, raw_y),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available()
        )

    sc_b53_te_X = apply_feature_scaler(split_te_X, scaler)
    test_loaders["B0053_test"] = DataLoader(
        NASABatteryDataset(sc_b53_te_X, split_te_y),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, test_loaders, scaler, (split_tr_y, split_te_y)


In [ ]:
# ==============================================================================
# SECTION 3: E06 SPLIT AUDIT & CRITICAL FAIRNESS CHECKS
# ==============================================================================
# Print Data Split Audit Table
print("=" * 85)
print(f"{'Cell':10s} | {'Train/Test':12s} | {'Evaluation Type':24s} | {'Cycles Used':28s}")
print("=" * 85)
print(f"{'B0005':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (168 cycles)':28s}")
print(f"{'B0006':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (168 cycles)':28s}")
print(f"{'B0007':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (168 cycles)':28s}")
print(f"{'B0029':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (40 cycles)':28s}")
print(f"{'B0030':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (40 cycles)':28s}")
print(f"{'B0031':10s} | {'TRAIN':12s} | {'N/A (Training Pool)':24s} | {'all available (40 cycles)':28s}")
print(f"{'B0018':10s} | {'TEST':12s} | {'unseen_cell':24s} | {'all available (132 cycles)':28s}")
print(f"{'B0032':10s} | {'TEST':12s} | {'unseen_cell':24s} | {'all available (39 cycles)':28s}")
print(f"{'B0053':10s} | {'TRAIN/TEST':12s} | {'temporal_extrapolation':24s} | {'0–36 (train) / 37–52 (test)':28s}")
print("=" * 85)

# Programmatic Split Verification
train_loader, test_loaders, scaler, (b53_tr_y, b53_te_y) = get_nasa_dataloaders(batch_size=8)
assert len(train_loader.dataset) == 660, f"Expected 660 train samples, got {len(train_loader.dataset)}"
assert len(test_loaders['B0018'].dataset) == 132, "B0018 sample count mismatch"
assert len(test_loaders['B0032'].dataset) == 39, "B0032 sample count mismatch"
assert len(test_loaders['B0053_test'].dataset) == 16, "B0053_test sample count mismatch"
assert len(b53_tr_y) == 37, f"Expected 37 B0053 train cycles (0-36), got {len(b53_tr_y)}"
assert len(b53_te_y) == 16, f"Expected 16 B0053 test cycles (37-52), got {len(b53_te_y)}"
assert len(b53_tr_y) + len(b53_te_y) == 53, "Total B0053 cycles mismatch (expected 53)"

# Critical Non-Leakage Assertions
assert "B0018" not in NASA_FULL_TRAIN_CELLS and "B0032" not in NASA_FULL_TRAIN_CELLS, "Test cell leakage in train pool!"
assert set(NASA_FULL_TRAIN_CELLS).isdisjoint(set(NASA_FULL_TEST_CELLS)), "Leakage: Train/Test cells overlap!"
assert scaler.n_samples_seen_ == 660 * SEQUENCE_LENGTH, f"Scaler saw {scaler.n_samples_seen_} points, expected exactly {660 * SEQUENCE_LENGTH}!"
print(f"[Fairness Audit] PASSED: Train={len(train_loader.dataset)} cycles, Test=187 cycles. Scaler fit ONLY on training data. Zero leakage verified.")


In [ ]:
# ==============================================================================
# SECTION 4: ARCHITECTURAL DEFINITIONS & MODEL REGISTRY
# (100% Faithful Line-by-Line Reproduction from baselineComparison.ipynb)
# ==============================================================================

# ==============================================================================
# PART A: ALL 10 ACTIVE BASELINE MODELS (FROM baselineComparison.ipynb)
# ==============================================================================
# ==============================================================================
# 4. ARCHITECTURAL DEFINITIONS FOR ALL 10 ACTIVE BASELINE MODELS
# (100% Faithful Line-by-Line Reproduction of Repository Baseline Suite)
# ==============================================================================

# ------------------------------------------------------------------------------
# Model 1: LSTM (Hochreiter & Schmidhuber, 1997)
# ------------------------------------------------------------------------------
"""Standard LSTM baseline model for battery SOH estimation."""


import torch
from torch import nn


class LSTMModel(nn.Module):
    """LSTM sequence model with input projection and regression head."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
    ) -> None:
        super().__init__()
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model) if use_projection else nn.Identity()
        lstm_in = d_model if use_projection else input_dim
        self.lstm = nn.LSTM(
            input_size=lstm_in,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4]
        x = self.input_projection(x)
        outputs, (h_n, _) = self.lstm(x)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            pooled = outputs[:, -1, :]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 2: GRU (Cho et al., 2014)
# ------------------------------------------------------------------------------
"""Standard GRU baseline model for battery SOH estimation."""


import torch
from torch import nn


class GRUModel(nn.Module):
    """GRU sequence model with input projection and regression head."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
    ) -> None:
        super().__init__()
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model) if use_projection else nn.Identity()
        gru_in = d_model if use_projection else input_dim
        self.gru = nn.GRU(
            input_size=gru_in,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        outputs, hidden = self.gru(x)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            pooled = outputs[:, -1, :]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 3: CNN1D (Kiranyaz et al., 2021)
# ------------------------------------------------------------------------------
"""1D Temporal Convolutional baseline model for battery SOH estimation."""


import torch
from torch import nn


class CNN1DModel(nn.Module):
    """1D CNN sequence model with multi-scale temporal convolutions and global pooling."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 3,
        kernel_size: int = 5,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        layers = []
        in_ch = input_dim
        for i in range(num_layers):
            out_ch = d_model
            layers.append(
                nn.Conv1d(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2,
                )
            )
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.GELU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch

        self.conv_net = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4] -> permute to [B, 4, L]
        x_conv = x.transpose(1, 2)
        features = self.conv_net(x_conv)  # [B, d_model, L]
        pooled = features.mean(dim=-1)     # Global average pooling -> [B, d_model]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 4: TCN (Bai, Kolter, & Koltun, 2018)
# ------------------------------------------------------------------------------
"""Temporal Convolutional Network (TCN) baseline model for battery SOH estimation."""


import torch
from torch import nn


class ChausalDilatedConv1DBlock(nn.Module):
    """Causal dilated conv block with residual connection."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, dilation: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=self.padding)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation, padding=self.padding)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Causal trim: drop trailing padding
        res = self.residual(x)
        out = self.conv1(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.drop1(self.act1(out))
        out = self.conv2(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.drop2(self.act2(out))
        return out + res


class TCNModel(nn.Module):
    """Deep Temporal Convolutional Network with exponential dilations."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        kernel_size: int = 3,
        num_levels: int = 4,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        layers = []
        in_ch = input_dim
        for i in range(num_levels):
            dilation = 2 ** i
            layers.append(
                ChausalDilatedConv1DBlock(
                    in_channels=in_ch,
                    out_channels=d_model,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = d_model

        self.network = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4] -> [B, 4, L]
        x_in = x.transpose(1, 2)
        feat = self.network(x_in)
        pooled = feat[:, :, -1]  # Last causal step
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 5: DLinear (Zeng et al., AAAI 2023)
# ------------------------------------------------------------------------------
"""DLinear baseline (LTSF-Linear family) adapted for NASA SOH seq-to-one regression.

Provenance (official upstream):
- Paper: "Are Transformers Effective for Time Series Forecasting?" (arXiv:2205.13504; AAAI 2023 per repo)
- Official repo: https://github.com/cure-lab/LTSF-Linear/
- Upstream file: models/DLinear.py
- Upstream commit (HEAD verified 2026-09-15): 0c113668a3b88c4c4ee586b8c5ec3e539c4de5a6
- License: Apache-2.0 (https://github.com/cure-lab/LTSF-Linear/blob/main/LICENSE)

Core architecture preserved? YES (series decomposition + linear seasonal/trend heads).
Task adaptation:
- Set pred_len = 1 (single-step output) and map the resulting channel vector to a scalar SOH via a small linear head.
"""


from dataclasses import dataclass

import torch
from torch import nn


# --------------------------------------------------------------------------------------
# Minimal upstream-derived core (Apache-2.0):
# This code is adapted from cure-lab/LTSF-Linear/models/DLinear.py with minimal changes.
# --------------------------------------------------------------------------------------


class _MovingAvg(nn.Module):
    """Moving average block to highlight the trend of time series (upstream: moving_avg)."""

    def __init__(self, kernel_size: int, stride: int = 1) -> None:
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.avg = nn.AvgPool1d(kernel_size=self.kernel_size, stride=stride, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, C]
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x_pad = torch.cat([front, x, end], dim=1)  # [B, L + pad, C]
        x_avg = self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)  # [B, L, C]
        return x_avg


class _SeriesDecomp(nn.Module):
    """Series decomposition block (upstream: series_decomp)."""

    def __init__(self, kernel_size: int) -> None:
        super().__init__()
        self.moving_avg = _MovingAvg(kernel_size, stride=1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean


@dataclass(frozen=True)
class DLinearConfig:
    seq_len: int = 512
    pred_len: int = 1
    enc_in: int = 4
    individual: bool = False
    kernel_size: int = 25  # upstream default


class _DLinearCore(nn.Module):
    """Upstream DLinear forward: [B, seq_len, C] -> [B, pred_len, C]."""

    def __init__(self, cfg: DLinearConfig) -> None:
        super().__init__()
        self.seq_len = cfg.seq_len
        self.pred_len = cfg.pred_len
        self.channels = cfg.enc_in
        self.individual = cfg.individual

        self.decomposition = _SeriesDecomp(cfg.kernel_size)

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList([nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)])
            self.Linear_Trend = nn.ModuleList([nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)])
        else:
            self.Linear_Seasonal = nn.Linear(self.seq_len, self.pred_len)
            self.Linear_Trend = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, C]
        seasonal_init, trend_init = self.decomposition(x)
        seasonal_init = seasonal_init.permute(0, 2, 1)  # [B, C, L]
        trend_init = trend_init.permute(0, 2, 1)        # [B, C, L]

        if self.individual:
            seasonal_output = torch.zeros(
                (seasonal_init.size(0), seasonal_init.size(1), self.pred_len),
                dtype=seasonal_init.dtype,
                device=seasonal_init.device,
            )
            trend_output = torch.zeros_like(seasonal_output)
            for i in range(self.channels):
                seasonal_output[:, i, :] = self.Linear_Seasonal[i](seasonal_init[:, i, :])
                trend_output[:, i, :] = self.Linear_Trend[i](trend_init[:, i, :])
        else:
            seasonal_output = self.Linear_Seasonal(seasonal_init)  # [B, C, pred_len]
            trend_output = self.Linear_Trend(trend_init)          # [B, C, pred_len]

        out = seasonal_output + trend_output  # [B, C, pred_len]
        return out.permute(0, 2, 1)  # [B, pred_len, C]


class DLinearSOHModel(nn.Module):
    """DLinear adapted to SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        kernel_size: int = 25,
        individual: bool = False,
        head: str = "linear",  # how to map channel vector -> scalar
    ) -> None:
        super().__init__()
        cfg = DLinearConfig(seq_len=seq_len, pred_len=1, enc_in=enc_in, individual=individual, kernel_size=kernel_size)
        self.core = _DLinearCore(cfg)

        if head == "mean":
            self.scalar_head = None
            self.head_mode = "mean"
        elif head == "linear":
            self.scalar_head = nn.Linear(enc_in, 1)
            self.head_mode = "linear"
        else:
            raise ValueError(f"Unknown head='{head}'. Use 'linear' or 'mean'.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[1] != 512 or x.shape[2] != 4:
            raise ValueError(f"Expected [B, 512, 4], got {tuple(x.shape)}")
        y_seq = self.core(x)              # [B, 1, 4]
        y_vec = y_seq[:, 0, :]            # [B, 4]
        if self.head_mode == "mean":
            y = y_vec.mean(dim=1, keepdim=True)
        else:
            y = self.scalar_head(y_vec)   # [B, 1]
        return y.squeeze(-1)


__all__ = ["DLinearSOHModel"]

# ------------------------------------------------------------------------------
# Model 6: Classical Transformer (Vaswani et al., 2017)
# ------------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

"""Classical Transformer Encoder baseline (matching nasa-te-q-transformer-transformer.ipynb)."""


import torch
from torch import nn


class TransformerModel(nn.Module):
    """Pure classical Transformer encoder baseline without quantum embedding."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        n_heads: int = 2,
        n_layers: int = 3,
        dim_feedforward: int = 64,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_cls_token: bool = True,
        pooling: str = "cls",
    ) -> None:
        super().__init__()
        self.use_cls_token = use_cls_token
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model)

        if use_cls_token:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.trunc_normal_(self.cls_token, std=0.02)
        else:
            self.cls_token = None

        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=1024)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4]
        h = self.input_projection(x)
        if self.use_cls_token and self.cls_token is not None:
            cls = self.cls_token.expand(x.size(0), -1, -1)
            h = torch.cat([cls, h], dim=1)
        h = self.positional_encoding(h)
        encoded = self.encoder(h)
        if self.use_cls_token and self.pooling == "cls":
            pooled = encoded[:, 0, :]
        else:
            pooled = encoded.mean(dim=1)
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 7: PatchTST (Nie et al., ICLR 2023)
# ------------------------------------------------------------------------------
"""PatchTST baseline adapted for NASA SOH seq-to-one regression.

Provenance (official upstream reference):
- Paper: "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers" (ICLR 2023; arXiv:2211.14730)
  - Paper URL: https://arxiv.org/abs/2211.14730
- Official repo: https://github.com/yuqinie98/PatchTST
- Upstream commit (HEAD verified 2026-09-15): 204c21efe0b39603ad6e2ca640ef5896646ab1a9
- License: Apache-2.0 (https://github.com/yuqinie98/PatchTST/blob/main/LICENSE)

Core architecture preserved? YES (patching + channel-independence + Transformer encoder).

Implementation note:
The official repo is a forecasting framework. Here we implement the **core PatchTST design**
(patching + channel-independence + Transformer encoder) and adapt only the task interface to
SOH regression:
- input: [B, 512, 4]
- output: scalar SOH [B]

Task adaptation (minimal):
- pred_len set to 1 (single-step output).
- per-channel outputs are fused to a scalar via a small linear head.
"""


from dataclasses import dataclass

import torch
from torch import nn


class _LearnablePositionalEncoding(nn.Module):
    """Learnable positional encoding (PatchTST uses learnable PE by default)."""

    def __init__(self, length: int, d_model: int) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, length, d_model))
        nn.init.trunc_normal_(self.pe, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class _RevIN(nn.Module):
    """Reversible Instance Normalization (RevIN), compact implementation."""

    def __init__(self, num_features: int, affine: bool = True, subtract_last: bool = False, eps: float = 1e-5) -> None:
        super().__init__()
        self.affine = affine
        self.subtract_last = subtract_last
        self.eps = eps
        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, num_features))
            self.beta = nn.Parameter(torch.zeros(1, 1, num_features))
        else:
            self.gamma = None
            self.beta = None
        self._last = None
        self._mean = None
        self._stdev = None

    def norm(self, x: torch.Tensor) -> torch.Tensor:
        if self.subtract_last:
            self._last = x[:, -1:, :].detach()
            x = x - self._last
        self._mean = x.mean(dim=1, keepdim=True).detach()
        x = x - self._mean
        self._stdev = torch.sqrt(torch.var(x, dim=1, keepdim=True, unbiased=False) + self.eps).detach()
        x = x / self._stdev
        if self.affine:
            x = x * self.gamma + self.beta
        return x

    def denorm(self, x: torch.Tensor) -> torch.Tensor:
        if self.affine:
            x = (x - self.beta) / (self.gamma + self.eps)
        x = x * self._stdev + self._mean
        if self.subtract_last:
            x = x + self._last
        return x


@dataclass(frozen=True)
class PatchTSTConfig:
    seq_len: int = 512
    enc_in: int = 4
    patch_len: int = 16
    stride: int = 8
    d_model: int = 64
    n_heads: int = 2
    e_layers: int = 3
    d_ff: int = 128
    dropout: float = 0.0
    revin: bool = True
    affine: bool = True
    subtract_last: bool = False


class PatchTSTSOHModel(nn.Module):
    """PatchTST (channel-independent) adapted to SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        patch_len: int = 16,
        stride: int = 8,
        d_model: int = 64,
        n_heads: int = 2,
        e_layers: int = 3,
        d_ff: int = 128,
        dropout: float = 0.0,
        revin: bool = True,
        affine: bool = True,
        subtract_last: bool = False,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        self.cfg = PatchTSTConfig(
            seq_len=seq_len,
            enc_in=enc_in,
            patch_len=patch_len,
            stride=stride,
            d_model=d_model,
            n_heads=n_heads,
            e_layers=e_layers,
            d_ff=d_ff,
            dropout=dropout,
            revin=revin,
            affine=affine,
            subtract_last=subtract_last,
        )

        patch_num = int((seq_len - patch_len) / stride + 1)
        self.revin = _RevIN(enc_in, affine=affine, subtract_last=subtract_last) if revin else None

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_enc = _LearnablePositionalEncoding(length=patch_num, d_model=d_model)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)

        # Per-channel head (pred_len = 1)
        self.channel_head = nn.Linear(d_model * patch_num, 1)

        # Channel fusion to scalar SOH
        self.scalar_head = nn.Sequential(
            nn.Linear(enc_in, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[1] != self.cfg.seq_len or x.shape[2] != self.cfg.enc_in:
            raise ValueError(f"Expected [B, {self.cfg.seq_len}, {self.cfg.enc_in}], got {tuple(x.shape)}")

        if self.revin is not None:
            x = self.revin.norm(x)

        # [B, L, C] -> [B, C, L] -> patches [B, C, P, PL]
        z = x.permute(0, 2, 1)
        patches = z.unfold(dimension=-1, size=self.cfg.patch_len, step=self.cfg.stride)
        B, C, P, PL = patches.shape
        tokens = patches.reshape(B * C, P, PL)  # channel-independent batch

        h = self.patch_embed(tokens)
        h = self.pos_enc(h)
        h = self.dropout(h)
        h = self.encoder(h)

        h_flat = h.reshape(B * C, -1)
        y_ch = self.channel_head(h_flat).reshape(B, C)  # [B, C]

        y = self.scalar_head(y_ch)  # [B, 1]
        return y.squeeze(-1)


__all__ = ["PatchTSTSOHModel"]

# ------------------------------------------------------------------------------
# Model 8: iTransformer (Liu et al., ICLR 2024 Spotlight)
# ------------------------------------------------------------------------------
"""iTransformer baseline adapted for NASA SOH seq-to-one regression.

Provenance (official upstream reference):
- Paper: "iTransformer: Inverted Transformers Are Effective for Time Series Forecasting" (ICLR 2024 Spotlight)
  - Paper PDF: https://proceedings.iclr.cc/paper_files/paper/2024/file/2ea18fdc667e0ef2ad82b2b4d65147ad-Paper-Conference.pdf
- Official repo: https://github.com/thuml/iTransformer
- Upstream commit (HEAD verified 2026-09-15): c2426e68ca13f74aaec08045c5c724d8ad328124
- License: MIT (per upstream repo)

Core architecture preserved? YES:
- **Inverted tokenization**: variates are tokens (N tokens), time points are token features.
- **Encoder-only Transformer**: native Transformer modules operate over variate tokens.

Task adaptation (minimal):
- Forecasting head replaced with a **seq-to-one regression head** for SOH.
- No timestamp covariates (`x_mark`) are used in this project; we follow upstream behavior for `x_mark=None`.
"""


from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ITransformerConfig:
    seq_len: int = 512
    enc_in: int = 4          # number of variates/tokens
    d_model: int = 64
    n_heads: int = 2
    e_layers: int = 3
    d_ff: int = 128
    dropout: float = 0.0
    use_norm: bool = True    # upstream-style per-sample normalization
    pooling: str = "mean"    # token pooling over variates


class _DataEmbeddingInverted(nn.Module):
    """Upstream DataEmbedding_inverted (simplified): linear map Time->d_model per variate token."""

    def __init__(self, c_in: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.value_embedding = nn.Linear(c_in, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, N] -> [B, N, L] -> [B, N, d_model]
        x = x.permute(0, 2, 1)
        x = self.value_embedding(x)
        return self.dropout(x)


class ITransformerSOHModel(nn.Module):
    """iTransformer-style inverted Transformer encoder for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        d_model: int = 64,
        n_heads: int = 2,
        e_layers: int = 3,
        d_ff: int = 128,
        dropout: float = 0.0,
        use_norm: bool = True,
        pooling: str = "mean",
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        self.cfg = ITransformerConfig(
            seq_len=seq_len,
            enc_in=enc_in,
            d_model=d_model,
            n_heads=n_heads,
            e_layers=e_layers,
            d_ff=d_ff,
            dropout=dropout,
            use_norm=use_norm,
            pooling=pooling,
        )

        # Embedding: invert and linearly embed per variate token
        self.enc_embedding = _DataEmbeddingInverted(c_in=seq_len, d_model=d_model, dropout=dropout)

        # Encoder-only Transformer over variate tokens (token length = enc_in = 4)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, N] where N=4
        if x.ndim != 3 or x.shape[1] != self.cfg.seq_len or x.shape[2] != self.cfg.enc_in:
            raise ValueError(f"Expected [B, {self.cfg.seq_len}, {self.cfg.enc_in}], got {tuple(x.shape)}")

        if self.cfg.use_norm:
            means = x.mean(dim=1, keepdim=True).detach()
            x0 = x - means
            stdev = torch.sqrt(torch.var(x0, dim=1, keepdim=True, unbiased=False) + 1e-5)
            x0 = x0 / stdev
        else:
            x0 = x

        # Embed and encode over variate tokens
        # embedding expects [B, L, N] but internally inverts to [B, N, L]
        enc_in = self.enc_embedding(x0)          # [B, N, d_model]
        enc_out = self.encoder(enc_in)           # [B, N, d_model]

        # Pool over tokens (variates)
        if self.cfg.pooling == "mean":
            pooled = enc_out.mean(dim=1)
        elif self.cfg.pooling == "cls":
            # optional: treat the first variate token as a representative token
            pooled = enc_out[:, 0, :]
        else:
            raise ValueError(f"Unknown pooling='{self.cfg.pooling}'.")

        y = self.head(pooled)  # [B, 1]
        return y.squeeze(-1)


__all__ = ["ITransformerSOHModel"]

# ------------------------------------------------------------------------------
# Model 9: QLSTM (Wang and Kebede hybrid QNN+LSTM baseline)
# ------------------------------------------------------------------------------
"""QLSTM baseline adapted from nasa-te-q-transformer-qlstm.ipynb.

Architecture (hybrid quantum-classical, NOT gate-level):
- Per-timestep linear projection: [B, L, 4] -> [B, L, n_qubits]
- PennyLane AngleEmbedding(Y) + BasicEntanglerLayers QNN (TorchLayer)
- Classical LSTM over quantum features
- Last/mean pooling + MLP head -> scalar SOH

Input/output contract matches all other E05 baselines: [B, 512, 4] -> [B].
"""


def _build_qnn_layer(n_qubits: int = 4, n_q_layers: int = 2):
    """Shared AngleEmbedding + BasicEntanglerLayers TorchLayer used by QLSTM and QNN-GRU."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_q_layers, n_qubits)}
    return qml.qnn.TorchLayer(qnode, weight_shapes)


class QLSTMModel(nn.Module):
    """Hybrid QNN front-end + classical LSTM for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        d_model: int = 64,
        hidden_size: int = 64,
        n_layers: int = 1,
        bidirectional: bool = False,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
        n_qubits: int = 4,
        n_q_layers: int = 2,
    ) -> None:
        super().__init__()
        if pooling not in {"last", "mean"}:
            raise ValueError("pooling must be either 'last' or 'mean'.")
        self.d_model = d_model
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.use_projection = use_projection
        self.n_qubits = n_qubits
        self.n_q_layers = n_q_layers
        self.qnn_in_dim = n_qubits
        self.qnn_device = torch.device("cpu")
        self.input_projection = nn.Linear(4, self.qnn_in_dim) if use_projection else nn.Identity()
        self.qnn_layer = _build_qnn_layer(n_qubits=n_qubits, n_q_layers=n_q_layers)
        self.qnn_layer.to(self.qnn_device)
        self.qnn_out_projection = nn.Linear(n_qubits, d_model)
        lstm_dropout = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=lstm_dropout,
        )
        pool_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(pool_dim, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        x = self.input_projection(x)
        q_dev = next(self.qnn_layer.parameters()).device
        x_q = x.reshape(batch_size * seq_len, -1).to(q_dev)
        qnn_features = self.qnn_layer(x_q)
        qnn_features = qnn_features.to(x.device)
        qnn_features = self.qnn_out_projection(qnn_features)
        qnn_features = qnn_features.view(batch_size, seq_len, -1)
        outputs, (h_n, _) = self.lstm(qnn_features)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            if self.bidirectional:
                forward_last = h_n[-2]
                backward_last = h_n[-1]
                pooled = torch.cat([forward_last, backward_last], dim=1)
            else:
                pooled = h_n[-1]
        return self.head(pooled).squeeze(-1)


__all__ = ["QLSTMModel"]

# ------------------------------------------------------------------------------
# Model 10: QNN-GRU (Soon and Soon hybrid QNN+GRU baseline)
# ------------------------------------------------------------------------------
"""QNN-GRU baseline adapted from nasa-te-q-transformer-qnn-gru.ipynb.

Architecture (hybrid quantum-classical, NOT gate-level):
- Per-timestep linear projection: [B, L, 4] -> [B, L, n_qubits]
- Shared PennyLane AngleEmbedding(Y) + BasicEntanglerLayers QNN (TorchLayer)
- Classical GRU over quantum features
- Last/mean pooling + MLP head -> scalar SOH

Input/output contract matches all other E05 baselines: [B, 512, 4] -> [B].
"""


class QNNGRUModel(nn.Module):
    """Hybrid QNN front-end + classical GRU for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        d_model: int = 64,
        hidden_size: int = 64,
        n_layers: int = 1,
        bidirectional: bool = False,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
        n_qubits: int = 4,
        n_q_layers: int = 2,
    ) -> None:
        super().__init__()
        if pooling not in {"last", "mean"}:
            raise ValueError("pooling must be either 'last' or 'mean'.")
        self.d_model = d_model
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.use_projection = use_projection
        self.n_qubits = n_qubits
        self.n_q_layers = n_q_layers
        self.qnn_in_dim = n_qubits
        self.qnn_device = torch.device("cpu")
        self.input_projection = nn.Linear(4, self.qnn_in_dim) if use_projection else nn.Identity()
        self.qnn_layer = _build_qnn_layer(n_qubits=n_qubits, n_q_layers=n_q_layers)
        self.qnn_layer.to(self.qnn_device)
        self.qnn_out_projection = nn.Linear(n_qubits, d_model)
        gru_dropout = dropout if n_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=gru_dropout,
        )
        pool_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(pool_dim, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        x = self.input_projection(x)
        q_dev = next(self.qnn_layer.parameters()).device
        x_q = x.reshape(batch_size * seq_len, -1).to(q_dev)
        qnn_features = self.qnn_layer(x_q)
        qnn_features = qnn_features.to(x.device)
        qnn_features = self.qnn_out_projection(qnn_features)
        qnn_features = qnn_features.view(batch_size, seq_len, -1)
        outputs, hidden = self.gru(qnn_features)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            if self.bidirectional:
                forward_last = hidden[-2]
                backward_last = hidden[-1]
                pooled = torch.cat([forward_last, backward_last], dim=1)
            else:
                pooled = hidden[-1]
        return self.head(pooled).squeeze(-1)


__all__ = ["QLSTMModel", "QNNGRUModel"]

print("[Models] All 10 baseline architectures successfully compiled.")

# ==============================================================================
# PART B: PROPOSED FOUNDATION MODEL: TE-Q-Transformer
# ==============================================================================
"""TE-Q-Transformer architecture copied from the NASA ablation notebook.

Source: notebooks/nasa/nasa_teq_component_ablation_study.ipynb model cell.
This module is used only for frozen NASA → CALCE inference. It does not retrain
NASA weights and is not a substitute for the original notebooks.
"""


from dataclasses import dataclass

import pennylane as qml
import torch
from torch import nn


@dataclass(frozen=True)
class TEQTransformerConfig:
    input_dim: int = 4
    seq_len: int = 512
    quantum_dim: int = 4
    d_model: int = 64
    n_heads: int = 2
    n_layers: int = 3
    dim_feedforward: int = 64
    dropout: float = 0.0
    q_device: str = "default.qubit"
    entangler_layers: int = 1
    entangler_type: str = "basic"
    use_pauli_feature_map: bool = False
    feature_map_reps: int = 1
    feature_map_entangle: bool = True
    use_cls_token: bool = True
    pooling: str = "cls"
    use_positional_encoding: bool = True
    use_temporal_smooth: bool = True
    temporal_kernel_size: int = 3
    use_gru_smoother: bool = False
    gru_num_layers: int = 1
    gru_dropout: float = 0.0
    head_hidden_dim: int = 64
    use_residual_mlp: bool = False
    residual_mlp_dim: int = 128


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class QuantumEmbeddingLayer(nn.Module):
    def __init__(
        self,
        n_qubits: int = 4,
        q_device: str = "default.qubit",
        entangler_layers: int = 1,
        entangler_type: str = "basic",
        use_pauli_feature_map: bool = False,
        feature_map_reps: int = 1,
        feature_map_entangle: bool = True,
    ) -> None:
        super().__init__()
        self.n_qubits = n_qubits
        self.R = 8.314462618
        self.T_ref = 298.15
        self.entangler_type = entangler_type
        self.use_pauli_feature_map = use_pauli_feature_map
        self.feature_map_reps = feature_map_reps
        self.feature_map_entangle = feature_map_entangle
        self.Ea_sei = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))
        self.Ea_pl = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))

        self.entangler_weights = nn.Parameter(
            0.01 * torch.randn(entangler_layers, n_qubits, dtype=torch.float32)
        )
        self.entangler_rzz = nn.Parameter(
            0.01 * torch.randn(entangler_layers, max(1, n_qubits - 1), dtype=torch.float32)
        )
        dev = qml.device(q_device, wires=n_qubits)

        def _apply_pauli_feature_map(inputs: torch.Tensor) -> None:
            for _ in range(self.feature_map_reps):
                for i in range(n_qubits):
                    qml.RX(inputs[:, i], wires=i)
                    qml.RY(inputs[:, i], wires=i)
                if self.feature_map_entangle:
                    for i in range(n_qubits - 1):
                        qml.CZ(wires=[i, i + 1])

        def _apply_basic_entangler(entangler_weights: torch.Tensor) -> None:
            qml.BasicEntanglerLayers(entangler_weights, wires=range(n_qubits))

        def _apply_rich_entangler(
            entangler_weights: torch.Tensor, entangler_rzz: torch.Tensor
        ) -> None:
            for layer in range(entangler_weights.shape[0]):
                for q in range(n_qubits):
                    qml.RZ(entangler_weights[layer, q], wires=q)
                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q + 1])
                for q in range(n_qubits - 1):
                    qml.IsingZZ(entangler_rzz[layer, q], wires=[q, q + 1])
                if n_qubits >= 3:
                    qml.Toffoli(wires=[0, 1, 2])
                    if n_qubits >= 4:
                        qml.Toffoli(wires=[1, 2, 3])
                for q in range(n_qubits):
                    qml.Hadamard(wires=q)
                    qml.RZ(entangler_weights[layer, q], wires=q)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(
            inputs: torch.Tensor,
            entangler_weights: torch.Tensor,
            entangler_rzz: torch.Tensor,
        ):
            if self.use_pauli_feature_map:
                _apply_pauli_feature_map(inputs)
            qml.RY(inputs[:, 0], wires=0)
            qml.RY(inputs[:, 1], wires=1)
            qml.RY(inputs[:, 2], wires=2)
            qml.RY(inputs[:, 3], wires=3)
            if self.entangler_type == "rich":
                _apply_rich_entangler(entangler_weights, entangler_rzz)
            else:
                _apply_basic_entangler(entangler_weights)
            return tuple(qml.expval(qml.PauliZ(i)) for i in range(n_qubits))

        self.circuit = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != 4:
            raise ValueError(f"Expected [B_flat, 4], got {tuple(x.shape)}")

        out_device = x.device
        out_dtype = x.dtype

        voltage_angle = x[:, 0] * torch.pi
        current_angle = x[:, 1] * torch.pi
        time_angle = x[:, 3] * torch.pi

        temp_c = x[:, 2]
        temp_k = torch.clamp(temp_c + 273.15, min=1.0)
        inv_t = 1.0 / temp_k
        inv_t_ref = 1.0 / self.T_ref
        Ea_sei_actual = self.Ea_sei * 10000.0
        Ea_pl_actual = self.Ea_pl * 10000.0

        sei_term = torch.exp((Ea_sei_actual / self.R) * (inv_t_ref - inv_t))
        plating_term = torch.exp((Ea_pl_actual / self.R) * (inv_t - inv_t_ref))
        phi = sei_term + plating_term
        theta_temp = torch.pi * phi / 4.0

        angles = torch.stack([voltage_angle, current_angle, time_angle, theta_temp], dim=1)

        angles_cpu = angles.to("cpu")
        entangler_cpu = self.entangler_weights.to("cpu")
        entangler_rzz_cpu = self.entangler_rzz.to("cpu")
        q_out = self.circuit(angles_cpu, entangler_cpu, entangler_rzz_cpu)
        q_tensor = torch.stack(q_out, dim=1).to(device=out_device, dtype=out_dtype)
        return q_tensor


class TEQTransformer(nn.Module):
    def __init__(self, cfg: TEQTransformerConfig | None = None) -> None:
        super().__init__()
        self.cfg = cfg or TEQTransformerConfig()
        if self.cfg.input_dim != 4:
            raise ValueError("This model expects exactly 4 input features.")
        if self.cfg.quantum_dim != 4:
            raise ValueError("quantum_dim must be 4 to match the 4 input channels.")
        if self.cfg.d_model % self.cfg.n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if self.cfg.d_model <= self.cfg.quantum_dim:
            raise ValueError("d_model should be larger than quantum_dim after projection.")
        if self.cfg.pooling not in {"cls", "mean"}:
            raise ValueError("pooling must be either 'cls' or 'mean'.")
        if self.cfg.use_cls_token and self.cfg.pooling != "cls":
            raise ValueError("use_cls_token=True requires pooling='cls'.")
        if not self.cfg.use_cls_token and self.cfg.pooling != "mean":
            raise ValueError("use_cls_token=False requires pooling='mean'.")
        if self.cfg.use_temporal_smooth and self.cfg.temporal_kernel_size % 2 == 0:
            raise ValueError(
                "temporal_kernel_size should be odd so the sequence length is preserved."
            )
        if self.cfg.use_gru_smoother and self.cfg.use_temporal_smooth:
            raise ValueError("Enable only one temporal module: GRU smoother or Conv1D smoother.")
        if self.cfg.entangler_type not in {"basic", "rich"}:
            raise ValueError("entangler_type must be 'basic' or 'rich'.")
        if self.cfg.head_hidden_dim <= 0:
            raise ValueError("head_hidden_dim must be positive.")

        self.quantum_embed = QuantumEmbeddingLayer(
            n_qubits=self.cfg.quantum_dim,
            q_device=self.cfg.q_device,
            entangler_layers=self.cfg.entangler_layers,
            entangler_type=self.cfg.entangler_type,
            use_pauli_feature_map=self.cfg.use_pauli_feature_map,
            feature_map_reps=self.cfg.feature_map_reps,
            feature_map_entangle=self.cfg.feature_map_entangle,
        )
        self.quantum_proj = nn.Linear(self.cfg.quantum_dim, self.cfg.d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, self.cfg.d_model)) if self.cfg.use_cls_token else None
        self.pos_encoder = (
            PositionalEncoding(self.cfg.d_model, max_len=self.cfg.seq_len + 2)
            if self.cfg.use_positional_encoding
            else None
        )

        if self.cfg.use_temporal_smooth:
            self.temporal_smooth = nn.Conv1d(
                in_channels=self.cfg.d_model,
                out_channels=self.cfg.d_model,
                kernel_size=self.cfg.temporal_kernel_size,
                padding=self.cfg.temporal_kernel_size // 2,
                bias=False,
            )
        else:
            self.temporal_smooth = nn.Identity()

        if self.cfg.use_gru_smoother:
            self.gru_smoother = nn.GRU(
                input_size=self.cfg.d_model,
                hidden_size=self.cfg.d_model,
                num_layers=self.cfg.gru_num_layers,
                dropout=self.cfg.gru_dropout if self.cfg.gru_num_layers > 1 else 0.0,
                batch_first=True,
            )
        else:
            self.gru_smoother = None

        self.residual_mlp = (
            nn.Sequential(
                nn.Linear(self.cfg.d_model, self.cfg.residual_mlp_dim),
                nn.GELU(),
                nn.Linear(self.cfg.residual_mlp_dim, self.cfg.d_model),
            )
            if self.cfg.use_residual_mlp
            else None
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.d_model,
            nhead=self.cfg.n_heads,
            dim_feedforward=self.cfg.dim_feedforward,
            dropout=self.cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.cfg.n_layers)
        self.head = nn.Sequential(
            nn.Linear(self.cfg.d_model, self.cfg.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(self.cfg.dropout),
            nn.Linear(self.cfg.head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        if seq_len != self.cfg.seq_len:
            raise ValueError(f"Expected sequence length {self.cfg.seq_len}, got {seq_len}.")

        x_flat = x.reshape(batch_size * seq_len, 4)
        q_features = self.quantum_embed(x_flat)
        q_features = self.quantum_proj(q_features)
        q_sequence = q_features.reshape(batch_size, seq_len, self.cfg.d_model)

        if self.gru_smoother is not None:
            q_sequence, _ = self.gru_smoother(q_sequence)

        q_sequence = q_sequence.transpose(1, 2)
        q_sequence = self.temporal_smooth(q_sequence)
        q_sequence = q_sequence.transpose(1, 2)

        if self.residual_mlp is not None:
            q_sequence = q_sequence + self.residual_mlp(q_sequence)

        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(batch_size, -1, -1)
            q_sequence = torch.cat([cls_tokens, q_sequence], dim=1)

        if self.pos_encoder is not None:
            q_sequence = self.pos_encoder(q_sequence)

        transformed = self.transformer(q_sequence)
        if self.cfg.pooling == "cls":
            pooled = transformed[:, 0]
        else:
            pooled = transformed.mean(dim=1)
        soh = self.head(pooled)
        return soh.squeeze(-1)


def rich_entangler_config() -> TEQTransformerConfig:
    """Config of the retained NASA ablation baseline `01_rich_entangler`."""
    return TEQTransformerConfig(entangler_type="rich")

# ==============================================================================
# PART C: UNIFIED E06 MODEL REGISTRY (ALL 11 MODELS)
# ==============================================================================
MODELS_REGISTRY: Dict[str, Dict[str, Any]] = {
    "TE-Q-Transformer": {
        "family": "Proposed-Quantum-Foundation",
        "builder": lambda: TEQTransformer(rich_entangler_config()),
        "expected_params": 92554,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "src/models/teq_transformer.py (Reference in baselineComparison.ipynb)",
        "e05_provenance": "Arrhenius SEI & Plating physics gates, Rich Entangler, CLS pooling",
    },
    "LSTM": {
        "family": "Recurrent",
        "builder": lambda: LSTMModel(),
        "expected_params": 71105,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 1: LSTM)",
        "e05_provenance": "Hochreiter & Schmidhuber (1997), 2-layer LSTM (hidden_dim=64, dropout=0.1)",
    },
    "GRU": {
        "family": "Recurrent",
        "builder": lambda: GRUModel(),
        "expected_params": 54465,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 2: GRU)",
        "e05_provenance": "Cho et al. (2014), 2-layer GRU (hidden_dim=64, dropout=0.1)",
    },
    "CNN1D": {
        "family": "Convolutional",
        "builder": lambda: CNN1DModel(),
        "expected_params": 47041,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 3: CNN1D)",
        "e05_provenance": "Kiranyaz et al. (2021), 4 conv blocks with dilation/stride, adaptive avg pool",
    },
    "TCN": {
        "family": "Convolutional",
        "builder": lambda: TCNModel(),
        "expected_params": 91841,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 4: TCN)",
        "e05_provenance": "Bai et al. (2018), causal dilated convolutions (1,2,4,8) with residual connections",
    },
    "DLinear": {
        "family": "Linear",
        "builder": lambda: DLinearSOHModel(),
        "expected_params": 1031,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 5: DLinear)",
        "e05_provenance": "Zeng et al. (AAAI 2023), moving avg series decomposition + seasonal & trend linear maps",
    },
    "Transformer": {
        "family": "Attention",
        "builder": lambda: TransformerModel(),
        "expected_params": 80257,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 6: Transformer)",
        "e05_provenance": "Vaswani et al. (2017), PositionalEncoding + 2-layer Multi-Head Attention Encoder (64 d_model)",
    },
    "PatchTST": {
        "family": "Attention",
        "builder": lambda: PatchTSTSOHModel(),
        "expected_params": 109962,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 7: PatchTST)",
        "e05_provenance": "Nie et al. (ICLR 2023), Channel-independent patch encoder + RevIN (affine=True)",
    },
    "iTransformer": {
        "family": "Attention",
        "builder": lambda: ITransformerSOHModel(),
        "expected_params": 137473,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 8: iTransformer)",
        "e05_provenance": "Liu et al. (ICLR 2024 Spotlight), Inverted variate tokenization across length 512",
    },
    "QLSTM": {
        "family": "Quantum-Recurrent",
        "builder": lambda: QLSTMModel(),
        "expected_params": 37853,
        "batch_size": 16,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 9: QLSTM)",
        "e05_provenance": "Wang & Kebede hybrid QNN+LSTM front-end (4 qubits, 2 layers) + LSTM",
    },
    "QNN-GRU": {
        "family": "Quantum-Recurrent",
        "builder": lambda: QNNGRUModel(),
        "expected_params": 29533,
        "batch_size": 16,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_script": "baselineComparison.ipynb (Model 10: QNN-GRU)",
        "e05_provenance": "Soon & Soon hybrid QNN+GRU front-end (4 qubits, 2 layers) + GRU",
    },
}

print(f"[Model Registry] Successfully registered all {len(MODELS_REGISTRY)} verified E06 models.")


In [ ]:
# ==============================================================================
# SECTION 5: 12-POINT PRE-FLIGHT TEST ACROSS ALL BENCHMARK MODELS
# ==============================================================================
print("=" * 75)
print(f"RUNNING 12-POINT PRE-FLIGHT TEST ACROSS ALL {len(MODELS_REGISTRY)} MODELS...")
print("=" * 75)

all_preflight_passed = True

for model_name, info in MODELS_REGISTRY.items():
    try:
        # 1. Instantiate
        m = info["builder"]().to(DEVICE)

        # 2. Count parameters
        tot_params = sum(p.numel() for p in m.parameters())
        trainable_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
        assert tot_params == info["expected_params"], (
            f"Param mismatch for {model_name}: expected {info['expected_params']}, got {tot_params}"
        )

        # 3. Load one dummy batch [B=2, L=512, D=4]
        bx = torch.randn(2, 512, 4, device=DEVICE)
        bx[:, :, 2] = 24.0  # Temperature channel in Celsius
        by = torch.tensor([0.95, 0.85], dtype=torch.float32, device=DEVICE)

        # 4 & 5. Forward pass
        m.train()
        out = m(bx)
        assert out.shape == (2,), f"Output shape mismatch: {out.shape}"
        assert torch.all(torch.isfinite(out)), "Non-finite output detected"

        # 6. Calculate loss
        crit = nn.MSELoss()
        loss = crit(out, by)
        assert torch.isfinite(loss), "Non-finite loss detected"

        # 7. Backward pass
        opt = optim.AdamW(m.parameters(), lr=1e-3)
        opt.zero_grad()
        loss.backward()
        for pname, p in m.named_parameters():
            if p.requires_grad and p.grad is not None:
                assert torch.all(torch.isfinite(p.grad)), f"Non-finite gradient in {pname}"

        # 8. Optimizer step
        opt.step()

        # 9 & 10. Save and reload checkpoint
        test_ckpt_path = CHECKPOINTS_DIR / f"test_sanity_{model_name.replace('-', '_')}.pth"
        torch.save(m.state_dict(), test_ckpt_path)
        m.load_state_dict(torch.load(test_ckpt_path, map_location=DEVICE))
        if test_ckpt_path.exists():
            test_ckpt_path.unlink()

        # 11. Perform prediction
        m.eval()
        with torch.no_grad():
            eval_out = m(bx)
        assert eval_out.shape == (2,), "Eval output shape mismatch"

        # 12. Calculate metrics
        metric_dict = compute_metrics(by.cpu().numpy(), eval_out.cpu().numpy())
        for k in ["RMSE", "MAE", "MAPE (%)", "R2", "MaxE"]:
            assert k in metric_dict and np.isfinite(metric_dict[k]), f"Invalid metric {k}"

        print(f"MODEL: {model_name:18s} | Params: {tot_params:7,d} | Status: PASS")

        # Cleanup test model
        del m, opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"MODEL: {model_name:18s} | Status: FAIL | Error: {e}")
        all_preflight_passed = False

print("=" * 75)
if all_preflight_passed:
    print(f"ALL {len(MODELS_REGISTRY)} MODELS PRE-FLIGHT TEST: PASS")
else:
    print("PRE-FLIGHT TEST: FAIL")
    raise RuntimeError("One or more models failed pre-flight verification! Execution halted.")
print("=" * 75)


In [ ]:
# ==============================================================================
# SECTION 6: COMMON REUSABLE TRAINING FUNCTION (E05 PROTOCOL)
# ==============================================================================
def train_model(
    model_name: str,
    model_info: Dict[str, Any],
    train_loader: DataLoader,
    device: torch.device,
    checkpoints_dir: Path,
    seed: int = 42,
    run_mode: str = "DRY_RUN",
) -> Tuple[Dict[str, torch.Tensor], List[Dict[str, Any]], int, float, float]:
    """Common standardized training function strictly adhering to the E05 protocol."""
    seed_everything(seed)
    model = model_info["builder"]().to(device)
    max_epochs = 1 if run_mode == "DRY_RUN" else model_info["max_epochs"]
    patience = model_info["patience"]
    lr = model_info["lr"]
    weight_decay = model_info["weight_decay"]

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    best_state = None
    ckpt_path = checkpoints_dir / f"E06_{model_name.replace('-', '_')}_best.pt"
    history_records: List[Dict[str, Any]] = []

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        epoch_t0 = time.time()
        model.train()
        running_loss = 0.0
        n_batches = 0

        for bx, by in train_loader:
            bx = bx.to(device)
            by = by.to(device)
            optimizer.zero_grad()
            preds = model(bx)
            loss = criterion(preds, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += float(loss.item())
            n_batches += 1
            if run_mode == "DRY_RUN":
                break  # Fast 1-batch execution in DRY_RUN mode

        epoch_loss = running_loss / max(n_batches, 1)
        epoch_time = time.time() - epoch_t0
        current_lr = optimizer.param_groups[0]["lr"]

        history_records.append({
            "model": model_name,
            "epoch": epoch,
            "loss": epoch_loss,
            "lr": current_lr,
            "epoch_time_sec": epoch_time,
            "seed": seed,
        })

        if run_mode == "FULL":
            scheduler.step(epoch_loss)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save({
                "epoch": epoch,
                "model_state": best_state,
                "loss": best_loss,
                "model_name": model_name,
                "seed": seed,
            }, ckpt_path)
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1 or epoch == max_epochs:
            print(f"[{model_name}] Epoch {epoch:2d}/{max_epochs} | Train Loss: {epoch_loss:.6f} | LR: {current_lr:.1e} | Best: {best_loss:.6f} (Ep {best_epoch}) | {epoch_time:.2f}s")

        if run_mode == "FULL" and patience_counter >= patience:
            print(f"[{model_name}] Early stopping triggered at epoch {epoch} (patience={patience}).")
            break

    total_training_sec = time.time() - start_time

    # Ensure best state is loaded
    try:
        best_state = torch.load(ckpt_path, map_location="cpu")
        if isinstance(best_state, dict) and "model_state" in best_state:
            best_state = best_state["model_state"]
    except Exception:
        best_state = model.state_dict()

    return best_state, history_records, best_epoch, best_loss, total_training_sec


In [ ]:
# ==============================================================================
# SECTION 7: SEQUENTIAL MODEL EXECUTION, EVALUATION & ARTIFACT EXPORT
# ==============================================================================
if RUN_MODE == "FULL":
    print("=" * 75)
    print(f"E06 FULL RUN STARTED ACROSS ALL {len(MODELS_REGISTRY)} MODELS")
    print("=" * 75)
else:
    print("=" * 75)
    print(f"E06 DRY RUN EXECUTION STARTED ACROSS ALL {len(MODELS_REGISTRY)} MODELS")
    print("=" * 75)

all_model_metrics: List[Dict[str, Any]] = []
all_cell_metrics: List[Dict[str, Any]] = []
all_predictions: List[Dict[str, Any]] = []
all_training_histories: List[Dict[str, Any]] = []
all_model_configs: List[Dict[str, Any]] = []
global_sample_idx = 0

# Try to capture git commit for provenance
try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    git_commit = "N/A"

execution_order = [
    "TE-Q-Transformer",
    "LSTM",
    "GRU",
    "CNN1D",
    "TCN",
    "DLinear",
    "Transformer",
    "PatchTST",
    "iTransformer",
    "QLSTM",
    "QNN-GRU",
]

for idx, model_name in enumerate(execution_order, start=1):
    model_info = MODELS_REGISTRY[model_name]
    print(f"\n{'#' * 75}")
    print(f"[{idx}/{len(execution_order)}] RUNNING: {model_name} ({model_info['family']})")
    print(f"{'#' * 75}")

    # 1. Get dataloaders for this model's specific batch size
    train_loader, test_loaders, scaler, _ = get_nasa_dataloaders(
        data_dir=DATA_ROOT, batch_size=model_info["batch_size"]
    )

    # 2. Train model sequentially
    best_state, history, best_epoch, best_loss, train_time = train_model(
        model_name=model_name,
        model_info=model_info,
        train_loader=train_loader,
        device=DEVICE,
        checkpoints_dir=CHECKPOINTS_DIR,
        seed=CURRENT_SEED,
        run_mode=RUN_MODE,
    )
    all_training_histories.extend(history)

    # 3. Instantiate model for evaluation and load best weights
    eval_model = model_info["builder"]().to(DEVICE)
    eval_model.load_state_dict(best_state)
    eval_model.eval()

    # 4. Evaluate on test splits
    infer_t0 = time.time()
    current_cell_records: List[Dict[str, Any]] = []

    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            eval_type = "unseen_cell" if cell_id in ("B0018", "B0032") else "temporal_extrapolation"
            y_t_list, y_p_list = [], []

            for bx, by in loader:
                bx = bx.to(DEVICE)
                pred = eval_model(bx)
                y_t_list.append(by.cpu().numpy().reshape(-1))
                y_p_list.append(pred.cpu().numpy().reshape(-1))

            yt_arr = np.concatenate(y_t_list)
            yp_arr = np.concatenate(y_p_list)
            m = compute_metrics(yt_arr, yp_arr)

            # Record per-cell metric
            cell_rec = {
                "model": model_name,
                "family": model_info["family"],
                "cell": cell_id,
                "eval_type": eval_type,
                "n_samples": len(yt_arr),
                "rmse": m["RMSE"],
                "mae": m["MAE"],
                "mape": m["MAPE (%)"],
                "r2": m["R2"],
                "max_error": m["MaxE"],
                "seed": CURRENT_SEED,
            }
            all_cell_metrics.append(cell_rec)
            current_cell_records.append(cell_rec)

            # Record individual predictions
            for t_val, p_val in zip(yt_arr, yp_arr):
                err = float(t_val - p_val)
                all_predictions.append({
                    "model": model_name,
                    "family": model_info["family"],
                    "cell": cell_id,
                    "eval_type": eval_type,
                    "sample_index": global_sample_idx,
                    "true_soh": float(t_val),
                    "pred_soh": float(p_val),
                    "error": err,
                    "abs_error": abs(err),
                    "squared_error": err ** 2,
                })
                global_sample_idx += 1

            print(f"  [{eval_type:22s}] {cell_id:12s} | N={len(yt_arr):3d} | RMSE: {m['RMSE']:.6f} | MAE: {m['MAE']:.6f} | R2: {m['R2']:.6f}")

    infer_time = time.time() - infer_t0

    # Overall macro metrics across the 3 evaluation splits
    macro_rmse = float(np.mean([c["rmse"] for c in current_cell_records]))
    macro_mae = float(np.mean([c["mae"] for c in current_cell_records]))
    macro_mape = float(np.mean([c["mape"] for c in current_cell_records]))
    macro_r2 = float(np.mean([c["r2"] for c in current_cell_records]))
    macro_max_e = float(np.max([c["max_error"] for c in current_cell_records]))

    tot_params = sum(p.numel() for p in eval_model.parameters())
    trainable_params = sum(p.numel() for p in eval_model.parameters() if p.requires_grad)

    model_metric_rec = {
        "model": model_name,
        "family": model_info["family"],
        "seed": CURRENT_SEED,
        "overall_rmse": macro_rmse,
        "overall_mae": macro_mae,
        "overall_mape": macro_mape,
        "overall_r2": macro_r2,
        "overall_max_error": macro_max_e,
        "trainable_params": trainable_params,
        "total_params": tot_params,
        "training_time_seconds": train_time,
        "inference_time_seconds": infer_time,
        "best_epoch": best_epoch,
        "device": str(DEVICE),
    }
    all_model_metrics.append(model_metric_rec)

    # Config record
    cfg_rec = {
        "model": model_name,
        "family": model_info["family"],
        "seed": CURRENT_SEED,
        "total_params": tot_params,
        "trainable_params": trainable_params,
        "batch_size": model_info["batch_size"],
        "max_epochs": model_info["max_epochs"],
        "patience": model_info["patience"],
        "learning_rate": model_info["lr"],
        "weight_decay": model_info["weight_decay"],
        "optimizer": "AdamW",
        "scheduler": "ReduceLROnPlateau(factor=0.5,patience=10)",
        "source_script": model_info["source_script"],
        "e05_provenance": model_info["e05_provenance"],
    }
    all_model_configs.append(cfg_rec)

    # Save individual report
    rep_text = f"""================================================================================
E06 RUN REPORT: {model_name}
================================================================================
Model: {model_name} ({model_info['family']})
Parameters: {tot_params:,} (Trainable: {trainable_params:,})
Device: {DEVICE} | Run Mode: {RUN_MODE} | Seed: {CURRENT_SEED}
Source Script: {model_info['source_script']}

OVERALL MACRO METRICS:
  RMSE: {macro_rmse:.6f}
  MAE:  {macro_mae:.6f}
  MAPE: {macro_mape:.4f}%
  R2:   {macro_r2:.6f}
  MaxE: {macro_max_e:.6f}

PER-CELL GENERALIZATION RESULTS:
"""
    for c in current_cell_records:
        rep_text += f"  - {c['cell']:12s} ({c['eval_type']:22s}) | RMSE: {c['rmse']:.6f} | MAE: {c['mae']:.6f} | R2: {c['r2']:.6f}\n"
    rep_text += f"\nTraining Time: {train_time:.2f}s (Best Epoch: {best_epoch})\nInference Time: {infer_time:.4f}s\n"
    (REPORTS_DIR / f"E06_{model_name.replace('-', '_')}_Run_Report.txt").write_text(rep_text, encoding="utf-8")

    # Save live CSVs after each model
    pd.DataFrame(all_model_metrics).to_csv(METRICS_DIR / "E06_model_metrics.csv", index=False)
    pd.DataFrame(all_cell_metrics).to_csv(METRICS_DIR / "E06_cell_metrics.csv", index=False)
    pd.DataFrame(all_predictions).to_csv(PREDICTIONS_DIR / "E06_predictions.csv", index=False)
    pd.DataFrame(all_training_histories).to_csv(TRAINING_DIR / "E06_training_history.csv", index=False)
    pd.DataFrame(all_model_configs).to_csv(CONFIGS_DIR / "E06_model_configs.csv", index=False)
    print(f"[Check] Saved live CSVs after {model_name} to {OUTPUT_ROOT}.")

# Final Master Artifacts
# 6. E06_provenance.json
provenance_meta = {
    "experiment": "E06_Generalization_Benchmark",
    "research_question": "Does TE-Q-Transformer performance remain consistent on unseen cells (B0018, B0032) and temporal extrapolation (B0053 70/30)?",
    "models_evaluated": list(MODELS_REGISTRY.keys()),
    "dataset": "NASA Ames Li-ion Battery Aging Dataset",
    "training_cells": list(NASA_FULL_TRAIN_CELLS) + ["B0053_first_70% (cycles 0-36)"],
    "unseen_test_cells": list(NASA_FULL_TEST_CELLS),
    "temporal_extrapolation_cell": "B0053_final_30% (cycles 37-52)",
    "training_samples": 660,
    "test_samples": 187,
    "sequence_length": 512,
    "features": ["V", "I", "T_C", "Time_norm"],
    "scaler_policy": "MinMaxScaler fitted STRICTLY on 660 training cycles; Temperature in unscaled Celsius",
    "seed": CURRENT_SEED,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    "git_commit": git_commit,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "run_mode": RUN_MODE,
}
(PROVENANCE_DIR / "E06_provenance.json").write_text(json.dumps(provenance_meta, indent=2), encoding="utf-8")
print(f"[Artifacts] Successfully wrote all standardized artifacts to {OUTPUT_ROOT.resolve()}")


In [ ]:
# ==============================================================================
# SECTION 8: AUTOMATIC METRIC RECONCILIATION
# ==============================================================================
print("=" * 75)
print("RUNNING AUTOMATIC METRIC RECONCILIATION...")
print("=" * 75)

preds_df = pd.read_csv(PREDICTIONS_DIR / "E06_predictions.csv")
cell_df = pd.read_csv(METRICS_DIR / "E06_cell_metrics.csv")
model_df = pd.read_csv(METRICS_DIR / "E06_model_metrics.csv")

recon_lines = [
    "================================================================================",
    "E06 AUTOMATIC RESULT RECONCILIATION REPORT",
    "================================================================================",
    f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    f"Total Predictions Loaded: {len(preds_df):,}",
    "================================================================================",
    "",
]

discrepancy_count = 0

for (m_name, c_id), group in preds_df.groupby(["model", "cell"]):
    recomp = compute_metrics(group["true_soh"].values, group["pred_soh"].values)
    matched = cell_df[(cell_df["model"] == m_name) & (cell_df["cell"] == c_id)].iloc[0]

    rmse_diff = abs(recomp["RMSE"] - matched["rmse"])
    mae_diff = abs(recomp["MAE"] - matched["mae"])
    r2_diff = 0.0 if np.isnan(recomp["R2"]) and np.isnan(matched["r2"]) else abs(recomp["R2"] - matched["r2"])

    if rmse_diff > 1e-5 or mae_diff > 1e-5 or r2_diff > 1e-5:
        discrepancy_count += 1
        msg = f"[DISCREPANCY] Model: {m_name}, Cell: {c_id} | RMSE diff: {rmse_diff:.2e}, MAE diff: {mae_diff:.2e}"
        print(f"WARNING: {msg}")
        recon_lines.append(f"WARNING: {msg}")
    else:
        recon_lines.append(f"MATCH: {m_name:18s} | Cell: {c_id:12s} | RMSE: {matched['rmse']:.6f} (recomputed diff < 1e-5)")

if discrepancy_count == 0:
    verdict = "ALL PREDICTIONS AND CELL METRICS ARE 100% MATHEMATICALLY RECONCILED."
    print(f"\nSUCCESS: {verdict}")
    recon_lines.append(f"\nVERDICT: PASS - {verdict}")
else:
    verdict = f"FOUND {discrepancy_count} DISCREPANCIES."
    print(f"\nWARNING: {verdict}")
    recon_lines.append(f"\nVERDICT: WARNING - {verdict}")

recon_text = "\n".join(recon_lines)
(REPORTS_DIR / "E06_Result_Reconciliation.txt").write_text(recon_text, encoding="utf-8")
print(f"[Reconciliation] Saved reconciliation report to {REPORTS_DIR / 'E06_Result_Reconciliation.txt'}")


In [ ]:
# ==============================================================================
# SECTION 9: GENERALIZATION SUMMARY TABLES (NUMERICAL COMPARISON)
# ==============================================================================
cell_df = pd.read_csv(METRICS_DIR / "E06_cell_metrics.csv")

# 1. Pivot Table: RMSE per Cell
rmse_pivot = cell_df.pivot(index="model", columns="cell", values="rmse")[["B0018", "B0032", "B0053_test"]]
rmse_pivot.columns = ["B0018 RMSE", "B0032 RMSE", "B0053-test RMSE"]

# 2. Pivot Table: R2 per Cell
r2_pivot = cell_df.pivot(index="model", columns="cell", values="r2")[["B0018", "B0032", "B0053_test"]]
r2_pivot.columns = ["B0018 R2", "B0032 R2", "B0053-test R2"]

# 3. Aggregate Generalization Breakdown:
#    - Unseen-cell mean RMSE: mean of (B0018, B0032)
#    - Unseen-cell RMSE std:  std of (B0018, B0032)
#    - Temporal extrapolation RMSE: B0053_test
summary_rows = []
for m_name in rmse_pivot.index:
    u_b18 = rmse_pivot.loc[m_name, "B0018 RMSE"]
    u_b32 = rmse_pivot.loc[m_name, "B0032 RMSE"]
    t_b53 = rmse_pivot.loc[m_name, "B0053-test RMSE"]
    unseen_mean = float(np.mean([u_b18, u_b32]))
    unseen_std = float(np.std([u_b18, u_b32]))
    summary_rows.append({
        "Model": m_name,
        "Unseen-cell mean RMSE": unseen_mean,
        "Unseen-cell RMSE std": unseen_std,
        "Temporal extrapolation RMSE": t_b53,
    })
gen_summary_df = pd.DataFrame(summary_rows).set_index("Model")

print("=" * 80)
print("TABLE 1: PER-CELL GENERALIZATION RMSE")
print("=" * 80)
print(rmse_pivot.to_string())
print("\n" + "=" * 80)
print("TABLE 2: PER-CELL GENERALIZATION R2")
print("=" * 80)
print(r2_pivot.to_string())
print("\n" + "=" * 80)
print("TABLE 3: AGGREGATE GENERALIZATION BREAKDOWN (UNSEEN-CELL VS TEMPORAL EXTRAPOLATION)")
print("=" * 80)
print(gen_summary_df.to_string())
print("=" * 80)


In [ ]:
# ==============================================================================
# SECTION 10: AUTO-ZIP ALL EXPERIMENT ARTIFACTS FOR 1-CLICK DOWNLOAD
# ==============================================================================
import shutil
import zipfile

zip_out_path = Path("E06_Generalization_Results.zip")
print(f"[Archive] Creating 1-click download bundle: {zip_out_path}...")

with zipfile.ZipFile(zip_out_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(OUTPUT_ROOT.parent)
            zipf.write(file_path, arcname)

print(f"[Archive] Successfully zipped all results to {zip_out_path.resolve()} ({zip_out_path.stat().st_size / 1e6:.2f} MB)")
print("You can download 'E06_Generalization_Results.zip' directly from the Kaggle Output viewer.")


In [ ]:
# ==============================================================================
# SECTION 11: COMPLETION STATUS
# ==============================================================================
if RUN_MODE == "DRY_RUN":
    print("\n" + "#" * 75)
    print("E06 DRY RUN: PASS")
    print(f"Environment, NASA data discovery, split audit, pre-flight checks across all {len(MODELS_REGISTRY)} models,")
    print("sequential training pipeline, prediction generation, metric reconciliation,")
    print("and artifact exports are 100% verified.")
    print("\nTo execute the FULL 80-epoch experiment on Kaggle GPU, change:")
    print("    RUN_MODE = 'FULL'")
    print("and run all cells.")
    print("#" * 75 + "\n")
else:
    print("\n" + "#" * 75)
    print("E06 FULL RUN COMPLETE: PASS")
    print(f"All {len(MODELS_REGISTRY)} benchmark models successfully trained and evaluated.")
    print("Full results preserved under E06_results/ directory and E06_Generalization_Results.zip.")
    print("#" * 75 + "\n")
